# IEEE-CIS: Random versus chronological holdout

## Summary

This notebook runs a **new independent full-data experiment** on the original IEEE-CIS training source. It compares the same restricted numeric features and 150-estimator LightGBM settings under stratified random and chronological 80/20 holdouts. All reported values below are computed during this run.

A difference between splits is not a causal estimate of leakage. The historical 600-estimator original notebook has separately been recovered; neither study replaces the published 500-estimator dashboard model.

**NEXUS / Fraud Intelligence** · Noncommercial historical research · CPU only · No customer decisions.

## Context and methods

- Seed 42; 150 estimators; learning rate 0.05; 31 leaves; two CPU threads.
- Exclude target, transaction ID and relative timestamp from predictors. Fit preprocessing has no label-dependent selection.
- Shared cutoff timestamps stay together in the chronological holdout.
- The masked card-family combination is a proxy, not a person or validated account identifier.
- Average precision depends on prevalence. Inspect holdout fraud rates alongside the metrics.
- Source SHA-256, environment versions and cohort counts are retained. No transaction or identity records are exported.

### Analysis implementation

The following standalone code is embedded from the repository comparison script. Its source SHA-256 is `8774d4e2d7869e04aac56f08350fe36dfa57cb2b7738b80fc00baa70156d7ad1`. No network download or credentials are needed inside the notebook.

In [ ]:
"""Reproduce IEEE-CIS random-vs-time holdout ranking, without model leakage.

This is a NEW comparison, not a reconstruction of the lost Kaggle notebook.
Requires the licensed Kaggle train_transaction.csv, not published in this repo.
"""

import argparse
import hashlib
import json
import platform
from pathlib import Path

import lightgbm
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
FEATURES = (
    ["TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1", "dist2"]
    + [f"C{i}" for i in range(1, 15)]
    + [f"D{i}" for i in range(1, 16)]
    + [f"V{i}" for i in range(1, 21)]
)
CARD_PROXY = ["card1", "card2", "card3", "card5", "addr1"]


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def split_indices(frame: pd.DataFrame, seed: int = SEED) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    if frame["isFraud"].nunique() != 2 or frame["TransactionDT"].isna().any():
        raise ValueError("Both fraud classes and nonnull TransactionDT are required")
    indices = np.arange(len(frame))
    random_train, random_test = train_test_split(
        indices, test_size=0.20, random_state=seed, stratify=frame["isFraud"].to_numpy()
    )
    # Don't split records sharing the cutoff timestamp across time windows.
    times = frame["TransactionDT"].to_numpy()
    sorted_times = np.sort(times)
    threshold = sorted_times[int(0.8 * len(frame))]
    time_train, time_test = indices[times < threshold], indices[times >= threshold]
    if not len(time_train) or not len(time_test):
        raise ValueError("Chronological holdout cannot separate timestamps")
    assert times[time_train].max() < times[time_test].min()
    return {"random": (random_train, random_test), "chronological": (time_train, time_test)}


def card_proxy(frame: pd.DataFrame) -> pd.Series:
    # Masked attributes are not verified customer IDs. This is only a card-family proxy.
    present = [x for x in CARD_PROXY if x in frame.columns]
    if not present:
        raise ValueError("Card-family proxy columns are missing")
    return frame[present].astype("string").fillna("<NA>").agg("|".join, axis=1)


def ranking_metrics(y: np.ndarray, scores: np.ndarray) -> dict:
    if len(y) == 0:
        return {"n": 0, "fraud_rate": None, "roc_auc": None, "average_precision": None}
    valid = len(np.unique(y)) == 2
    return {
        "n": int(len(y)),
        "fraud_rate": round(float(np.mean(y)), 6),
        "roc_auc": round(float(roc_auc_score(y, scores)), 6) if valid else None,
        "average_precision": round(float(average_precision_score(y, scores)), 6) if valid else None,
    }


def compare(frame: pd.DataFrame, seed: int = SEED) -> dict:
    feature_names = [x for x in FEATURES if x in frame.columns]
    if not feature_names:
        raise ValueError("No allowed numeric predictors were found")
    X = frame[feature_names].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = frame["isFraud"].astype(int).to_numpy()
    group = card_proxy(frame).to_numpy()
    splits = split_indices(frame, seed)
    result = {}
    for name, (fit, holdout) in splits.items():
        model = LGBMClassifier(
            n_estimators=150, learning_rate=0.05, num_leaves=31,
            random_state=seed, n_jobs=2, verbosity=-1,
        )
        model.fit(X.iloc[fit], y[fit])
        scores = model.predict_proba(X.iloc[holdout])[:, 1]
        seen = np.isin(group[holdout], np.unique(group[fit]))
        result[name] = {
            "train_rows": int(len(fit)),
            "holdout_rows": int(len(holdout)),
            "holdout": ranking_metrics(y[holdout], scores),
            "seen_card_proxy_fraction": round(float(np.mean(seen)), 6),
            "seen_proxy_subset": ranking_metrics(y[holdout][seen], scores[seen]),
            "unseen_proxy_subset": ranking_metrics(y[holdout][~seen], scores[~seen]),
            "train_max_transaction_dt": float(frame["TransactionDT"].iloc[fit].max()),
            "holdout_min_transaction_dt": float(frame["TransactionDT"].iloc[holdout].min()),
        }
    return {
        "study": "IEEE-CIS random and chronological holdout, new reproduction",
        "seed": seed,
        "predictors": feature_names,
        "rows": int(len(frame)),
        "splits": result,
        "interpretation_limit": (
            "A random-vs-time score difference is not by itself a causal estimate of leakage; "
            "temporal distribution shift and card-family correlation can also affect it. "
            "Card attributes are masked proxies, not validated customer IDs."
        ),
    }


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--csv", type=Path, required=True, help="Original Kaggle train_transaction.csv")
    parser.add_argument("--output", type=Path, default=Path("outputs/research/ieee_cis_split_comparison.json"))
    parser.add_argument("--max-rows", type=int, default=0, help="Optional smoke subset; never call this full-data")
    args = parser.parse_args(argv)
    if args.max_rows < 0:
        parser.error("--max-rows must be >= 0")
    needed = {"isFraud", "TransactionDT", *FEATURES, *CARD_PROXY}
    df = pd.read_csv(args.csv, usecols=lambda c: c in needed, nrows=args.max_rows or None)
    if not {"isFraud", "TransactionDT"}.issubset(df.columns):
        raise ValueError("CSV missing required isFraud / TransactionDT columns")
    result = compare(df)
    result["input_sha256"] = file_sha256(args.csv)
    result["input_path_logged"] = False
    result["subset_mode"] = bool(args.max_rows)
    result["versions"] = {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "lightgbm": lightgbm.__version__,
        "sklearn": sklearn.__version__,
    }
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(result, indent=2))
    return 0

## Data and execution

Attach the **IEEE-CIS Fraud Detection competition** data after accepting the competition terms. This run requires exactly 590,540 training transactions. `TransactionDT` is relative elapsed seconds, not a real date. No partial sample is accepted as full-data evidence.

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt

candidates = sorted(Path('/kaggle/input').rglob('train_transaction.csv'))
assert len(candidates) == 1, 'Attach exactly one original IEEE-CIS competition train_transaction.csv'
source_file = candidates[0]
report_path = Path('/kaggle/working/ieee_cis_split_comparison.json')
main(['--csv', str(source_file), '--output', str(report_path)])
report = json.loads(report_path.read_text())
assert report['rows'] == 590_540 and not report['subset_mode'], 'Full source required'
assert report['splits']['chronological']['train_max_transaction_dt'] < report['splits']['chronological']['holdout_min_transaction_dt']


## Results

These tables and figures reflect this executed run. Metrics from the GitHub historical study are not copied into the calculation.

In [ ]:
summary = pd.DataFrame({name: values['holdout'] for name, values in report['splits'].items()}).T
summary.index.name = 'Holdout strategy'
display(summary.rename(columns={'n':'Holdout rows','fraud_rate':'Fraud prevalence','roc_auc':'ROC-AUC','average_precision':'Average precision'}).round(6))

colors = ['#4EA8FF', '#6E55C6']
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
for ax, metric, title in zip(axes, ['roc_auc','average_precision'], ['ROC-AUC by holdout strategy','Average precision by holdout strategy']):
    bars = ax.bar(summary.index, summary[metric], color=colors)
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.set_ylabel('Score (higher is better)')
    ax.bar_label(bars, fmt='%.4f', padding=4)
    ax.spines[['top','right']].set_visible(False)
fig.suptitle('IEEE-CIS full training source · independent held-out evaluation', fontsize=12)
fig.savefig('/kaggle/working/holdout_comparison.png', dpi=160)
plt.show()


### Seen and unseen masked proxies

Proxy recurrence can differ between splits. This stratification is observational: it neither establishes customer identity nor separates a causal time effect from selection, prevalence and distribution shift.

In [ ]:
cohorts = []
for strategy, values in report['splits'].items():
    for key, label in [('seen_proxy_subset','Seen proxy'),('unseen_proxy_subset','Unseen proxy')]:
        cohorts.append({'Strategy':strategy,'Cohort':label,**values[key]})
cohort_table = pd.DataFrame(cohorts)
display(cohort_table.round(6))
fig, ax = plt.subplots(figsize=(9, 4.2), constrained_layout=True)
cohort_table.pivot(index='Strategy', columns='Cohort', values='average_precision').plot.bar(ax=ax,color=colors,rot=0)
ax.set_ylim(0,1);ax.set_ylabel('Average precision');ax.set_title('Held-out ranking by observed masked-proxy recurrence')
ax.spines[['top','right']].set_visible(False)
fig.savefig('/kaggle/working/proxy_cohort_comparison.png',dpi=160)
plt.show()


## Takeaways and reproducibility

Interpret the observed random/chronological gap as a diagnostic. This experiment does not estimate prevented losses, prove causal leakage or calibrate a deployable fraud probability. New data, a matched model contract and independent evaluation are required for model promotion.

In [ ]:
random_metrics = report['splits']['random']['holdout']
time_metrics = report['splits']['chronological']['holdout']
display(Markdown(f"**Measured ROC-AUC gap (random minus chronological): {random_metrics['roc_auc']-time_metrics['roc_auc']:+.6f}.** "
                 f"Average precision: random {random_metrics['average_precision']:.6f}, chronological {time_metrics['average_precision']:.6f}. "
                 "These are different holdout populations, not a causal decomposition."))
display(pd.DataFrame([{'Training rows':report['rows'],'Seed':report['seed'],'Subset':report['subset_mode'],'Input SHA-256':report['input_sha256']}]))
display(pd.DataFrame.from_dict(report['versions'],orient='index',columns=['Version']))
